In [ ]:
%load_ext autoreload
%autoreload 2
import jax
import jax.numpy as jnp
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import src.spaces as spaces

spaces.SETUP = spaces.make_setup_uv()

from src.training import (
    train, loss_fn, loss_fn_fa, loss_fn_ea, loss_fn_da, DEFAULT_CONFIG,
)
from src.teacher import make_wi_particles, make_arbitrary_particles, sample_data
from src.metrics import rmd_to_projected
from src.utils import plot_rmd_curves, plot_particles_3d, plot_uv_equivariance

import matplotlib.pyplot as plt

N_VALUES = [5, 10, 50, 100, 500]
N_REPS = 6
CONFIG = {**DEFAULT_CONFIG, "T": 15.0, "gr": 5}
CONFIG["alpha"]=5.0
CONFIG['tau'] = 0.0

In [ ]:
print(CONFIG)

In [ ]:
teacher_wi = make_wi_particles()

results_wi = {"vanilla": [], "DA": [], "FA": [], "EA": []}

for N in N_VALUES:
    for name in results_wi:
        results_wi[name].append([])

    for rep in range(N_REPS):
        key = jax.random.PRNGKey(rep * 1000 + N)

        # Vanilla
        h = train(teacher_wi, N, jax.random.fold_in(key, 0), CONFIG, "si")
        results_wi["vanilla"][-1].append(rmd_to_projected(h["particles"][-1]))

        # DA
        h = train(teacher_wi, N, jax.random.fold_in(key, 1), CONFIG, "si", used_loss_fn=loss_fn_da)
        results_wi["DA"][-1].append(rmd_to_projected(h["particles"][-1]))

        # FA
        h = train(teacher_wi, N, jax.random.fold_in(key, 2), CONFIG, "si", used_loss_fn=loss_fn_fa)
        results_wi["FA"][-1].append(rmd_to_projected(h["particles"][-1]))

        # EA
        h = train(teacher_wi, N, jax.random.fold_in(key, 3), CONFIG, "si", used_loss_fn=loss_fn_ea)
        results_wi["EA"][-1].append(rmd_to_projected(h["particles"][-1]))

    print(f"N={N} done")

In [ ]:
plot_rmd_curves(N_VALUES, results_wi, "Number of Particles (N)", "Distance to E^G — WI teacher, SI init (uv)", save_pdf=True)


In [ ]:
teacher_arb = make_arbitrary_particles()

results_arb = {"vanilla": [], "DA": [], "FA": [], "EA": []}

for N in N_VALUES:
    for name in results_arb:
        results_arb[name].append([])

    for rep in range(N_REPS):
        key = jax.random.PRNGKey(rep * 1000 + N)

        h = train(teacher_arb, N, jax.random.fold_in(key, 0), CONFIG, "si")
        results_arb["vanilla"][-1].append(rmd_to_projected(h["particles"][-1]))

        h = train(teacher_arb, N, jax.random.fold_in(key, 1), CONFIG, "si", used_loss_fn=loss_fn_da)
        results_arb["DA"][-1].append(rmd_to_projected(h["particles"][-1]))

        h = train(teacher_arb, N, jax.random.fold_in(key, 2), CONFIG, "si", used_loss_fn=loss_fn_fa)
        results_arb["FA"][-1].append(rmd_to_projected(h["particles"][-1]))

        h = train(teacher_arb, N, jax.random.fold_in(key, 3), CONFIG, "si", used_loss_fn=loss_fn_ea)
        results_arb["EA"][-1].append(rmd_to_projected(h["particles"][-1]))

    print(f"N={N} done")

In [ ]:
plot_rmd_curves(N_VALUES, results_arb, "Number of Particles (N)", "Distance to E^G — arbitrary teacher, SI init(uv)", save_pdf=True)

In [ ]:
teacher = make_wi_particles()
N = 500
key = jax.random.PRNGKey(42)
CONFIG['T'] = 20.0
h_vanilla = train(teacher, N, jax.random.fold_in(key, 0), CONFIG, "si")
print("vanilla done")
# h_da = train(teacher, N, jax.random.fold_in(key, 1), CONFIG, "si", used_loss_fn=loss_fn_da)
# print("DA done")
# h_fa = train(teacher, N, jax.random.fold_in(key, 2), CONFIG, "si", used_loss_fn=loss_fn_fa)
# print("FA done")
# h_ea = train(teacher, N, jax.random.fold_in(key, 3), CONFIG, "si", used_loss_fn=loss_fn_ea)
# print("EA done")

In [ ]:
plot_uv_equivariance(h_vanilla["particles"][-1], teacher, title="Student vs teacher — vanilla", filename="uv_vanilla_WI.pdf", save_pdf=True)


In [ ]:
plot_uv_equivariance(h_da["particles"][-1], teacher, title="Student vs teacher — DA")


In [ ]:
plot_uv_equivariance(h_fa["particles"][-1], teacher, title="Student vs teacher — FA")


In [ ]:
plot_uv_equivariance(h_ea["particles"][-1], teacher, title="Student vs teacher — EA")


In [ ]:
from src.teacher import make_arbitrary_particles

teacher_arb = make_arbitrary_particles()
CONFIG['T'] = 20.0
key = jax.random.PRNGKey(42)
N=500
h_vanilla_arb = train(teacher_arb, N, jax.random.fold_in(key, 0), CONFIG, "si")
print("vanilla done")
# h_da_arb = train(teacher_arb, N, jax.random.fold_in(key, 1), CONFIG, "si", used_loss_fn=loss_fn_da)
# print("DA done")
# h_fa_arb = train(teacher_arb, N, jax.random.fold_in(key, 2), CONFIG, "si", used_loss_fn=loss_fn_fa)
# print("FA done")
# h_ea_arb = train(teacher_arb, N, jax.random.fold_in(key, 3), CONFIG, "si", used_loss_fn=loss_fn_ea)
# print("EA done")

In [ ]:
plot_uv_equivariance(h_vanilla_arb["particles"][-1], teacher_arb, title="Arbitrary teacher — vanilla", filename="uv_vanilla_arb.pdf", save_pdf=True)


In [ ]:
plot_uv_equivariance(h_da_arb["particles"][-1], teacher_arb, title="Arbitrary teacher — DA")


In [ ]:
plot_uv_equivariance(h_fa_arb["particles"][-1], teacher_arb, title="Arbitrary teacher — FA")


In [ ]:
plot_uv_equivariance(h_ea_arb["particles"][-1], teacher_arb, title="Arbitrary teacher — EA")
